In [ ]:
# Make cesc importable in this kernel. Locates setup_env.py by walking
# up from the working directory, so this cell works from the package
# root, from examples/, or any subfolder. Importing setup_env puts
# src/ on the path. If you ran `python setup_env.py` once, a plain
# `import cesc` works and this is harmless.
import sys, pathlib
sys.path.insert(0, str(next(
    p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    if (p / 'setup_env.py').exists())))
import setup_env

import cesc
print('cesc', cesc.__version__)

# cesc quickstart

A synthetic case, start to finish, in under five minutes: a stable-plus-PI environment on the west half of a small domain, one reflectivity cell inside it, one outside. The algorithm keeps the supported cell and drops the other at the gate.

In [ ]:
import numpy as np
import cesc
from cesc import adapters
from cesc.environment import scan_pi_stable

nz, ny, nx = 20, 120, 160
z = np.linspace(0, 6000, nz)
heights = np.broadcast_to(z[:, None, None], (nz, ny, nx)).copy()
theta = np.broadcast_to((280 + 0.0005 * z)[:, None, None], (nz, ny, nx)).copy()
rise = np.clip(z / 1500, 0, 1) * 4.0
drop = np.where(z > 2000, -3.0 * np.clip((z - 2000) / 1000, 0, 1), 0.0)
theta[:, :, : nx // 2] = (280 + rise + drop)[:, None, None]

Z = np.full((ny, nx), 8.0)
Z[50:60, 30:40] = 30.0     # inside the supportive environment
Z[50:60, 130:140] = 30.0   # outside it

ds = adapters.from_arrays(
    reflectivity=Z, theta_e=theta, heights=heights,
    u3d=np.full((nz, ny, nx), 10.0), v3d=np.zeros((nz, ny, nx)),
    dx_km=0.9, dy_km=0.9)
ds

Stage 1 scan, then the full pipeline for the one timestep.

In [ ]:
p = cesc.Params(env_mode="ECE")
da = ds.isel(time=0)
dth = np.diff(da.theta_e.values, axis=0)
pi, _, st, _, h0, h1, _, _ = scan_pi_stable(
    dth, da.heights.values, p.max_pi_height_m, p.unstable_threshold,
    p.stable_threshold, p.min_pi_depth_m, p.max_stable_height_m, p.theta_eps)

rate = np.zeros((ny, nx)); rate[52:58, 32:38] = 2.0
out = cesc.identify_time(da.reflectivity_1km, pi, st, h0, h1,
                         u3d=da.u3d, v3d=da.v3d, heights=da.heights,
                         precip_rate=rate, params=p)
cesc.print_summary(out["summary"])

Three panels: reflectivity with object contours, the environment and corridor, attributed precipitation.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].imshow(Z, origin="lower", cmap="viridis"); ax[0].set_title("Z + kept objects")
ax[0].contour(out["labels_env_da"].values > 0, colors="r", linewidths=1.5)
ax[1].imshow(out["masks"]["support"].values, origin="lower", cmap="Greys")
ax[1].contour(out["masks"]["ece_core"].values, colors="b", linewidths=1)
ax[1].set_title("support mask (grey) and core (blue)")
cr = out["conv_rate"].values
ax[2].imshow(np.where(np.isfinite(cr), cr, 0), origin="lower", cmap="Blues")
ax[2].set_title("attributed rate (mm/h)")
plt.tight_layout()

The per-object table:

In [ ]:
import pandas as pd
pd.DataFrame([o for o in out["objects"]])[
    ["id", "kept_env", "intensity_cls", "size_km2", "frac_within_support",
     "frac_strong", "cls"]]